# cable Batch Workflow

批量入口 notebook：扫描一个目录下第一层的主配置 JSON，展开其中引用的全部模板，并逐个运行 `cable` compare。


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start=None):
    cwd = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "examples" / "neuron_compare" / "cable" / "workflow_api.py").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


REPO_ROOT = find_repo_root()
CABLE_ROOT = REPO_ROOT / "examples" / "neuron_compare" / "cable"
if str(CABLE_ROOT) not in sys.path:
    sys.path.insert(0, str(CABLE_ROOT))

import workflow_api


## Parameters

`config_dir` 只扫描第一层 `*.json` 主配置。每个主配置里的每个模板引用都会展开成一个 runnable item。


In [ ]:
config_dir = CABLE_ROOT / "configs"
plot_cases = False
expand_only = False
summary_dir = None


## Discover Runnable Config x Template Runs

In [ ]:
config_records = workflow_api.discover_batch_configs(config_dir)
preview_df = pd.DataFrame(
    [
        {
            "run_id": row["run_id"],
            "config_name": row["config_name"],
            "template_name": row["template_name"],
            "config_path": str(row["config_path"]),
            "template_path": str(row["template_path"]),
            "group_id": row["group_id"],
            "n_expanded_cases": row["n_expanded_cases"],
            "default_out_dir": str(row["default_out_dir"]),
        }
        for row in config_records
    ]
)
display(preview_df)


## Run Batch Workflow

In [ ]:
batch_result = workflow_api.run_notebook_batch(
    config_records,
    plot=plot_cases,
    expand_only=expand_only,
    summary_dir=summary_dir,
)
print("summary_dir:", batch_result["summary_dir"])
print("manifest_path:", batch_result["manifest_path"])
print("config_runs_path:", batch_result["config_runs_path"])
print("observable_summary_path:", batch_result["observable_summary_path"])
print("failures_path:", batch_result["failures_path"])


In [ ]:
tables = workflow_api.build_batch_summary_tables(batch_result)
display(tables["runs_df"])
display(tables["observables_df"])
display(tables["failures_df"])
